In [2]:
s = "Processing ENSG00000174059 ...[1394/2036] ✓ Completed ENSG00000174021 (948 variants)"
s.split()[5]

'ENSG00000174021'

In [1]:
import os
import pandas as pd


In [14]:
betas_dir = "/gpfs/commons/home/adas/uTWAS/src/results/baseline_full/bayesian_ridge/betas"
beta_file = os.path.join(betas_dir, "chr1", "ENSG00000285629.tsv.gz")

betas = pd.read_csv(beta_file, sep="\t", index_col=0)

betas



,beta
variant_id,
1:6059549_G_A,0.000602
1:6059683_G_A,-0.000968
1:6059745_G_A,-0.001188
1:6060051_A_G,0.000749
1:6060194_A_C,-0.000037
...,...
1:6297290_G_A,0.001647
1:6297414_T_C,0.001654
1:6297477_G_A,0.000521


In [10]:
df = pd.read_csv("/gpfs/commons/home/adas/uTWAS/src/results/baseline_full/bayesian_ridge/chr21.tsv", sep="\t", index_col=0)

In [12]:
df.loc['ENSG00000141956']['alpha']

np.float64(3.546121253677222)

In [1]:
"""
apply_zscore_rare_variants.py

Example: apply the saved scaler to rare variant ChromBPNet VEP scores.

For each (cell_type, assay, score):
  1. Load scored variant-peak pairs across all chromosomes
  2. Per variant: pick the row with max |score|, preserving sign
  3. Normalize using the saved scaler from zscore_scalers.joblib
     (StandardScaler with_mean=False: divides by std, sign preserved)

Output: one parquet per cell type with columns:
  CHR, SNP, BP, A1, A2, assay, score_name, raw_score, zscore
"""

import pandas as pd
import numpy as np
import os
import sys
import joblib
from tqdm import tqdm

CHR_NUM = sys.argv[1]

ANNOTATION_DIR = '/gpfs/commons/groups/knowles_lab/vmazeeva/BigBrain/Processed/annotations/'


# ─── Constants ───────────────────────────────────────────────────────────────
meta_cols = ['CHR', 'SNP', 'BP', 'A1', 'A2']
cell_type_list = ['microglia', 'astrocyte', 'neuron', 'oligodendrocyte']
assay_list = ['ATAC', 'H3K27ac', 'H3K4me3']
chrs = list(range(1, 23))

score_cols = [
    'log_counts_diff_chrombpnet',
    'log_probs_diff_abs_sum_chrombpnet',
    'probs_jsd_diff_chrombpnet',
]

short_names = {
    'log_counts_diff_chrombpnet': 'log_counts_diff',
    'log_probs_diff_abs_sum_chrombpnet': 'log_probs_diff_abs',
    'probs_jsd_diff_chrombpnet': 'probs_jsd_diff',
}

base_dir = '/gpfs/commons/groups/knowles_lab/data/ADSP_reguloML/annotations_hg38/merged_annotations_ADSP_v2'
chrombpnet_dir = f'{base_dir}/chrombpnet/variant_peak_pairs_scored'
scaler_dir = f'{chrombpnet_dir}/zscore_scaler'

scored_dir = '/gpfs/commons/groups/knowles_lab/data/ADSP_reguloML/ADSP_vcf/58K_preview_compact/rare_variants/chrombpnet/variant_peak_pairs_scored'
out_dir = os.path.join(scored_dir, 'zscore_normalized')

# ─── Load scalers ────────────────────────────────────────────────────────────
scalers_path = os.path.join(scaler_dir, 'zscore_scalers.joblib')
print(f'Loading scalers from {scalers_path}...')
scalers = joblib.load(scalers_path)

Loading scalers from /gpfs/commons/groups/knowles_lab/data/ADSP_reguloML/annotations_hg38/merged_annotations_ADSP_v2/chrombpnet/variant_peak_pairs_scored/zscore_scaler/zscore_scalers.joblib...


/gpfs/commons/home/vmazeeva/.local/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.1.2 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [2]:
scalers

{'microglia__ATAC__log_counts_diff': StandardScaler(with_mean=False),
 'microglia__ATAC__log_probs_diff_abs': StandardScaler(with_mean=False),
 'microglia__ATAC__probs_jsd_diff': StandardScaler(with_mean=False),
 'microglia__H3K27ac__log_counts_diff': StandardScaler(with_mean=False),
 'microglia__H3K27ac__log_probs_diff_abs': StandardScaler(with_mean=False),
 'microglia__H3K27ac__probs_jsd_diff': StandardScaler(with_mean=False),
 'microglia__H3K4me3__log_counts_diff': StandardScaler(with_mean=False),
 'microglia__H3K4me3__log_probs_diff_abs': StandardScaler(with_mean=False),
 'microglia__H3K4me3__probs_jsd_diff': StandardScaler(with_mean=False),
 'astrocyte__ATAC__log_counts_diff': StandardScaler(with_mean=False),
 'astrocyte__ATAC__log_probs_diff_abs': StandardScaler(with_mean=False),
 'astrocyte__ATAC__probs_jsd_diff': StandardScaler(with_mean=False),
 'astrocyte__H3K27ac__log_counts_diff': StandardScaler(with_mean=False),
 'astrocyte__H3K27ac__log_probs_diff_abs': StandardScaler(wit